In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# Selecting 1 image for subject

In this section we will select just one image per person. So we will build a subset for CW and DeepFool attack.

In [ ]:
import os
import shutil
import random

# ==============================================================================
# Impostazione dei percorsi
# ==============================================================================
SOURCE_DIR = './dataset/dataset_100_subjects'
DEST_DIR = './dataset/dataset_1_per_person'

# Crea la directory di destinazione (se esiste già, non fa nulla)
os.makedirs(DEST_DIR, exist_ok=True)

print("Inizio la selezione di un'immagine per soggetto...")

# Contatore per tenere traccia delle immagini elaborate
processed_count = 0

# ==============================================================================
# Estrazione e copia
# ==============================================================================
# Scansiona tutte le cartelle all'interno del dataset locale
for subject_folder in os.listdir(SOURCE_DIR):
    subject_path = os.path.join(SOURCE_DIR, subject_folder)

    # Verifica che sia effettivamente una cartella
    if os.path.isdir(subject_path):
        # Filtra solo i file immagine
        images = [f for f in os.listdir(subject_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        if images:
            selected_image = random.choice(images)

            src_image_path = os.path.join(subject_path, selected_image)

            # Estrae l'estensione originale (.jpg, .png, ecc.)
            ext = os.path.splitext(selected_image)[1]

            # Rinomina il file con l'ID del soggetto per mantenere la tracciabilità assoluta
            new_filename = f"{subject_folder}{ext}"
            dest_image_path = os.path.join(DEST_DIR, new_filename)

            # Copia il file nella nuova destinazione
            shutil.copy2(src_image_path, dest_image_path)
            processed_count += 1

print(f"\nOperazione completata! Sono state estratte {processed_count} immagini.")
print(f"Le puoi trovare nella cartella: {DEST_DIR}")

In [19]:
import os
import numpy as np
import pandas as pd
import time
from PIL import Image

# Import per il modello e il preprocessing
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
import tensorflow as tf

# Import ART
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import CarliniLInfMethod

# ==============================================================================
# 0. CONFIGURAZIONE DEL TEAM E DEI PERCORSI
# ==============================================================================
# OGNUNO DEI 3 MEMBRI DEVE IMPOSTARE IL PROPRIO ID QUI (1, 2, o 3)
WORKER = ""

# --- PERCORSI (Verifica che siano corretti per il tuo PC) ---
DATA_DIR = './dataset/dataset_1_per_person'
RESULTS_DIR = './results'
CSV_PATH = './dataset/selected_100_subjects.csv'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Il range visivo è [0.0, 1.0]. Il vincolo del 10% è esattamente 0.1.
EPSILON_LIMIT = 0.1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"--- AVVIO SCRIPT - LAVORATORE {WORKER} su {device} ---")

# ==============================================================================
# FASE 1: CARICAMENTO DEL MODELLO E DEI LABEL (CON MAPPATURA CSV)
# ==============================================================================
print("\n[Fase 1] Caricamento del modello Target NN1 (InceptionResnetV1)...")
nn1 = InceptionResnetV1(pretrained='vggface2').eval().to(device)
nn1.classify = True

print("Downloading official VGGFace2 label mapping...")
fpath = tf.keras.utils.get_file(
    'rcmalli_vggface_labels_v2.npy',
    "https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_labels_v2.npy",
    cache_subdir="./"
)
LABELS = np.load(fpath)

print("Lettura del CSV e sincronizzazione degli ID in corso...")
df_subjects = pd.read_csv(CSV_PATH)

df_subjects['Class_ID'] = df_subjects['Class_ID'].astype(str).str.strip()
df_subjects['Name'] = df_subjects['Name'].astype(str).str.strip()

label_to_index = {}

for _, row in df_subjects.iterrows():
    c_id = row['Class_ID']
    p_name = row['Name']

    found_idx = -1
    for idx, lbl in enumerate(LABELS):
        lbl_str = str(lbl).strip().strip("'").strip('"')
        if c_id in lbl_str or p_name in lbl_str:
            found_idx = idx
            break

    if found_idx != -1:
        label_to_index[c_id] = found_idx
    else:
        print(f"ERRORE (Ignorabile): Impossibile trovare '{c_id}' o '{p_name}'")

print(f"Mappatura completata con successo. Trovati {len(label_to_index)} soggetti validi.")

# ==============================================================================
# FASE 2: PREPROCESSING E CARICAMENTO IMMAGINI (RANGE [0, 1])
# ==============================================================================
print("\n[Fase 2] Preprocessing e Caricamento delle immagini...")

# Manteniamo le immagini in [0, 1]. ART gestirà la normalizzazione per il modello.
preprocess_attack = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
])

x_data = []
y_labels_numeric = []

image_files = sorted([f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))])

if len(image_files) != 100:
    print(f"ATTENZIONE: Trovate {len(image_files)} immagini invece di 100!")

for img_name in image_files:
    img_path = os.path.join(DATA_DIR, img_name)

    img = Image.open(img_path).convert('RGB')
    img_tensor = preprocess_attack(img)
    x_data.append(img_tensor.numpy())

    class_id = os.path.splitext(img_name)[0].strip()

    if class_id in label_to_index:
        y_labels_numeric.append(label_to_index[class_id])
    else:
        print(f"Errore critico: ID {class_id} non mappato.")
        y_labels_numeric.append(0)

x_data = np.array(x_data)

num_classes = len(LABELS)
y_data_one_hot = np.eye(num_classes)[y_labels_numeric]

print(f"Caricate {len(x_data)} immagini. Shape: {x_data.shape}")

# ==============================================================================
# FASE 3: WRAPPER ART E SUDDIVISIONE (SLICING)
# ==============================================================================
print("\n[Fase 3] Configurazione ART Classifier...")

loss_fn = torch.nn.CrossEntropyLoss()

# Diciamo ad ART che l'input è [0, 1] e gli diamo i parametri per normalizzare in [-1, 1] internamente
classifier = PyTorchClassifier(
    model=nn1,
    clip_values=(0.0, 1.0),
    loss=loss_fn,
    optimizer=None,
    input_shape=(3, 160, 160),
    nb_classes=num_classes,
    preprocessing=(np.array([0.5, 0.5, 0.5]), np.array([0.5, 0.5, 0.5])),
    device_type='gpu' if torch.cuda.is_available() else 'cpu'
)

if WORKER == "Christian":
    start_idx, end_idx = 0, 2
elif WORKER == "Salvatore":
    start_idx, end_idx = 30, 60
elif WORKER == "Candido":
    start_idx, end_idx = 60, 100
else:
    raise ValueError("WORKER non valido. Scegli Christian, Tartaglione o Candido.")

x_test_subset = x_data[start_idx:end_idx]
y_test_subset = y_data_one_hot[start_idx:end_idx]

print(f"Lavoratore {WORKER}: assegnate {len(x_test_subset)} immagini (indici {start_idx} - {end_idx-1}).")

preds_clean = np.argmax(classifier.predict(x_test_subset), axis=1)
true_labels = np.argmax(y_test_subset, axis=1)
correct_clean_mask = (preds_clean == true_labels)
num_correct_clean = np.sum(correct_clean_mask)

print(f"Accuratezza iniziale su questo subset: {num_correct_clean}/{len(x_test_subset)} classificate correttamente.")

# ==============================================================================
# FASE 4: GRID SEARCH CARLINI & WAGNER L_INF (CON STAMPA PER IMMAGINE)
# ==============================================================================
print("\n[Fase 4] Avvio attacco C&W L_inf (con clipping al 10% -> eps=0.1)...")

max_iter_list = [25, 50, 100]
learning_rate_list = [0.01, 0.05]
confidence_list = [0.0, 0.5, 1.0]

results = []
total_combinations = len(max_iter_list) * len(learning_rate_list) * len(confidence_list)
current_combo = 1

for m_iter in max_iter_list:
    for lr in learning_rate_list:
        for conf in confidence_list:
            print(f"\n[{current_combo}/{total_combinations}] Configurazione: max_iter={m_iter}, lr={lr}, confidence={conf}")

            start_time_combo = time.time()

            attack_cw = CarliniLInfMethod(
                classifier=classifier,
                targeted=False,
                max_iter=m_iter,
                learning_rate=lr,
                confidence=conf,
                verbose=False
            )

            x_adv_raw_list = []
            print(f"   -> Generazione in corso per {len(x_test_subset)} immagini...")

            for i in range(len(x_test_subset)):
                start_img_time = time.time()

                # Generazione attacco singola immagine
                single_adv = attack_cw.generate(x=x_test_subset[i:i+1])

                img_time = time.time() - start_img_time
                print(f"      - Immagine {i+1:02d}/{len(x_test_subset)} processata in {img_time:.2f} secondi")

                x_adv_raw_list.append(single_adv)

            x_adv_raw = np.concatenate(x_adv_raw_list, axis=0)

            # CLIPPING: La perturbazione non può superare 0.1 nel range originale
            perturbation = x_adv_raw - x_test_subset
            perturbation_clipped = np.clip(perturbation, -EPSILON_LIMIT, EPSILON_LIMIT)

            # Riapplichiamo la perturbazione tagliata e limitiamo al range visivo [0.0, 1.0]
            x_adv_clipped = np.clip(x_test_subset + perturbation_clipped, 0.0, 1.0)

            # Valutazione sulle immagini compromesse
            preds_adv = np.argmax(classifier.predict(x_adv_clipped), axis=1)
            success_mask = correct_clean_mask & (preds_adv != true_labels)

            if num_correct_clean > 0:
                asr = (np.sum(success_mask) / num_correct_clean) * 100
            else:
                asr = 0.0

            mean_linf_raw = np.mean(np.max(np.abs(perturbation), axis=(1, 2, 3)))

            elapsed_time_combo = time.time() - start_time_combo

            results.append({
                'WORKER': WORKER,
                'max_iter': m_iter,
                'learning_rate': lr,
                'confidence': conf,
                'ASR_Clipped (%)': round(asr, 2),
                'Mean_Linf_Original': round(mean_linf_raw, 4),
                'Time_Elapsed (s)': round(elapsed_time_combo, 2)
            })

            print(f" --> RISULTATO CONFIGURAZIONE: ASR: {asr:.2f}% | L_inf iniziale: {mean_linf_raw:.4f} | Tempo Totale: {elapsed_time_combo:.1f}s")
            current_combo += 1

# ==============================================================================
# FASE 5: SALVATAGGIO DEI DATI DEL LAVORATORE
# ==============================================================================
df_results = pd.DataFrame(results)
csv_filename = f'cw_grid_search_worker_{WORKER}.csv'
csv_filepath = os.path.join(RESULTS_DIR, csv_filename)

df_results.to_csv(csv_filepath, index=False)

print("\n=======================================================")
print(f"Lavoro completato! Risultati salvati in:\n{csv_filepath}")
print("=======================================================")

--- AVVIO SCRIPT - LAVORATORE Christian su cuda ---

[Fase 1] Caricamento del modello Target NN1 (InceptionResnetV1)...


C:\Users\chris\PycharmProjects\AI4Cyber_Group15\.venv\Lib\site-packages\facenet_pytorch\models\inception_resnet_v1.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  stat

Lettura del CSV e sincronizzazione degli ID in corso...
ERRORE (Ignorabile): Impossibile trovare 'n004459' o 'Julio_César_Chávez,_Jr.'
Mappatura completata con successo. Trovati 99 soggetti validi.

[Fase 2] Preprocessing e Caricamento delle immagini...
Errore critico: ID n004459 non mappato.
Caricate 100 immagini. Shape: (100, 3, 160, 160)

[Fase 3] Configurazione ART Classifier...
Lavoratore Christian: assegnate 2 immagini (indici 0 - 1).
Accuratezza iniziale su questo subset: 2/2 classificate correttamente.

[Fase 4] Avvio attacco C&W L_inf (con clipping al 10% -> eps=0.1)...

[1/18] Configurazione: max_iter=25, lr=0.01, confidence=0.0
   -> Generazione in corso per 2 immagini...
      - Immagine 01/2 processata in 187.32 secondi
      - Immagine 02/2 processata in 123.15 secondi
 --> RISULTATO CONFIGURAZIONE: ASR: 100.00% | L_inf iniziale: 0.0100 | Tempo Totale: 310.5s

[2/18] Configurazione: max_iter=25, lr=0.01, confidence=0.5
   -> Generazione in corso per 2 immagini...
      - 

KeyboardInterrupt: 